In [1]:
"""
=============================================================================
PAPER : AdaDrift-SRI — Adaptive Drift-Aware Online Learning Algorithm
         for Non-Stationary Data Streams


LEAKAGE CONTROLS (ALL VERIFIED):
  ✅ Target (SRI) excluded from feature set
  ✅ High_Risk_Regime excluded
  ✅ StandardScaler fitted on warm-up partition ONLY (30%)
  ✅ Prequential protocol: each observation seen EXACTLY ONCE (test-then-train)
  ✅ No temporal overlap between warm-up and evaluation
  ✅ SWRT-XGB scaler: fit ONCE on warm-up, transform only during eval [FIXED]

OUTPUTS (IEEE-compliant):
  - Table I: Dataset Characteristics
  - Table II: Algorithm Performance Comparison
  - Table III: Annual Performance Breakdown
  - Table IV: Drift Events Summary
  - Table V: Crisis Resilience Analysis
  - Figure 1: Cumulative R² Progression
  - Figure 2: Prediction vs Actual Time Series (Full Timeline)
  - Figure 3: Error Distribution QQ-Plot
  - Figure 4: Drift Detection Timeline
  - Figure 5: Rolling RMSE Monitoring
  - Figure 6: Crisis Period Analysis
  - Figure 7: Model Ranking Radar Chart
  - Figure 8: Computational Efficiency

USAGE (Jupyter Notebook):
    DATA_PATH = 'dataset.csv'
    OUTPUT_DIR = './output/'
    WARMUP_PCT = 0.30

REQUIREMENTS:
    pip install river xgboost scikit-learn pandas numpy scipy matplotlib seaborn
=============================================================================
"""
import os, warnings, time
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════
DATA_PATH = 'dataset.csv'
OUTPUT_DIR = './output/'
WARMUP_PCT = 0.30

os.makedirs(OUTPUT_DIR, exist_ok=True)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import stats
from math import pi

from river import tree as rtree, preprocessing as rprep
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb
np.random.seed(42)

from IPython.display import display, HTML

# ─── IEEE STYLE CONFIGURATION ─────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
    'lines.linewidth': 1.2,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
})

IEEE_COLORS = {
    'AdaDrift-SRI': '#1f77b4',
    'Static-XGB': '#ff7f0e',
    'Online-SGD': '#2ca02c',
    'SWRT-XGB': '#d62728',
    'Crisis': '#9467bd',
    'Drift': '#e377c2',
}

def display_table(df, title, save_path=None):
    """Display styled DataFrame in Jupyter and save to CSV"""
    styled = df.style\
        .set_caption(title)\
        .set_table_styles([
            {'selector': 'caption', 'props': [
                ('font-family', 'Times New Roman'),
                ('font-size', '12pt'),
                ('font-weight', 'bold'),
                ('text-align', 'center'),
                ('margin-bottom', '10px')
            ]},
            {'selector': 'th', 'props': [
                ('font-family', 'Times New Roman'),
                ('font-size', '10pt'),
                ('font-weight', 'bold'),
                ('background-color', '#1f77b4'),
                ('color', 'white'),
                ('text-align', 'center'),
                ('padding', '8px 12px'),
                ('border', '1px solid #ddd')
            ]},
            {'selector': 'td', 'props': [
                ('font-family', 'Times New Roman'),
                ('font-size', '10pt'),
                ('text-align', 'center'),
                ('padding', '6px 10px'),
                ('border', '1px solid #ddd')
            ]},
            {'selector': 'tr:nth-child(even)', 'props': [
                ('background-color', '#f2f2f2')
            ]},
        ])\
        .set_properties(**{'text-align': 'center'})
    display(styled)
    if save_path:
        df.to_csv(save_path, index=False)
        print(f"  >> Saved: {save_path}")

def print_plain_table(df, title):
    """Print table in plain text format for paper"""
    print(f"\n{'='*80}")
    print(f"  {title}")
    print(f"{'='*80}")
    print(df.to_string(index=False))
    print(f"{'='*80}\n")

def save_figure(fig, filename):
    """Save figure in PDF + PNG for IEEE submission"""
    fig.savefig(f'{OUTPUT_DIR}{filename}.pdf', format='pdf', dpi=300, bbox_inches='tight')
    fig.savefig(f'{OUTPUT_DIR}{filename}.png', format='png', dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  >> {filename}.pdf / .png")

print("="*70)
print("AdaDrift-SRI: Adaptive Drift-Aware Online Learning Algorithm")
print(f"Data: {DATA_PATH} | Output: {OUTPUT_DIR}")
print("="*70)

# ═══════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════════════════════
print("\n[LOADING DATA]...")
df = pd.read_csv(DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
banks = ['BBRI','BBTN','BMRI','BBNI']

FEATURES = []
for b in banks:
    FEATURES += [f'{b}_Volatility',f'{b}_Return',f'{b}_LogReturn',
                 f'{b}_BB_Width',f'{b}_ROC_10',f'{b}_Momentum_10']
FEATURES += ['Avg_Volatility','Avg_Correlation','Max_Correlation',
             'Avg_Correlation_lag1','Avg_Correlation_lag2','Avg_Correlation_lag3',
             'Crisis_Intensity_30d']
for b in banks:
    for k in range(1,3): FEATURES.append(f'{b}_Return_lag{k}')

# Leakage check
for f in FEATURES:
    assert 'SRI' not in f and 'High_Risk' not in f, f"LEAKAGE: {f}"
print(f"[LEAKAGE CHECK] PASS — {len(FEATURES)} features, no target leakage")

TARGET = 'Systemic_Risk_Index'
df_c = df[FEATURES+[TARGET,'Crisis_Period','Date','Year']].dropna().reset_index(drop=True)
N = len(df_c)
WARMUP = int(N * WARMUP_PCT)

X_all = df_c[FEATURES].values
y_all = df_c[TARGET].values
dates = df_c['Date'].values
crisis = df_c['Crisis_Period'].values
years = df_c['Year'].values

X_wu = X_all[:WARMUP]
y_wu = y_all[:WARMUP]
X_ev = X_all[WARMUP:]
y_ev = y_all[WARMUP:]
cr_ev = crisis[WARMUP:]
dt_ev = dates[WARMUP:]
yr_ev = years[WARMUP:]

# StandardScaler: fit ONLY on warm-up
sc = StandardScaler()
sc.fit(X_wu)
X_wu_sc = sc.transform(X_wu)
X_ev_sc = sc.transform(X_ev)

print(f"[DATA] N={N}, Warm-up={WARMUP} ({WARMUP/N*100:.0f}%), Eval={N-WARMUP}")
print(f"[DATES] {pd.Timestamp(dates[0]).date()} to {pd.Timestamp(dates[-1]).date()}")

# ═══════════════════════════════════════════════════════════════════════════
# TABLE I: Dataset Characteristics
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("TABLE I: Dataset Characteristics")
print("="*70)

table1_data = {
    'Parameter': [
        'Total Observations',
        'Warm-up Period',
        'Evaluation Period',
        'Feature Count (Total)',
        'Feature Count (Bank-Specific)',
        'Feature Count (Market-Wide)',
        'Target Variable',
        'Crisis Period Observations',
        'Training Strategy',
        'Date Range (Warm-up)',
        'Date Range (Evaluation)',
        'Banks Covered'
    ],
    'Value': [
        f'{N:,}',
        f'{WARMUP:,} ({WARMUP_PCT*100:.0f}%)',
        f'{N-WARMUP:,} ({(1-WARMUP_PCT)*100:.0f}%)',
        f'{len(FEATURES)}',
        f'{sum(1 for f in FEATURES if any(b in f for b in banks))}',
        f'{sum(1 for f in FEATURES if not any(b in f for b in banks))}',
        'Systemic Risk Index (SRI)',
        f'{(cr_ev==1).sum():,} ({(cr_ev==1).sum()/len(cr_ev)*100:.1f}%)',
        'Prequential (test-then-train)',
        f'{pd.Timestamp(dates[0]).date()} to {pd.Timestamp(dates[WARMUP-1]).date()}',
        f'{pd.Timestamp(dates[WARMUP]).date()} to {pd.Timestamp(dates[-1]).date()}',
        ', '.join(banks)
    ]
}
df_t1 = pd.DataFrame(table1_data)
display_table(df_t1, 'TABLE I: Dataset Characteristics', f'{OUTPUT_DIR}table1_dataset_characteristics.csv')
print_plain_table(df_t1, 'TABLE I: Dataset Characteristics')

# ═══════════════════════════════════════════════════════════════════════════
# ALGORITHM CLASSES
# ═══════════════════════════════════════════════════════════════════════════

class PHDetector:
    """C1: Page-Hinkley Drift Detector. O(1) per step."""
    def __init__(self, delta=0.005, lam=40., alpha=0.9999):
        self.delta = delta
        self.lam = lam
        self.alpha = alpha
        self.sum_ = 0.
        self.mu = 0.
    
    def update(self, x):
        self.mu = self.alpha * self.mu + (1 - self.alpha) * x
        self.sum_ = max(0., self.sum_ + x - self.mu - self.delta)
        return self.sum_ > self.lam
    
    def reset(self):
        self.sum_ = 0.


class AdaDriftSRI:
    """
    AdaDrift-SRI: Adaptive Drift-Aware Online Learning Algorithm.
    C1: Page-Hinkley Detector | C2: HATR | C3: SW Evaluator | C4: Warm-Restart
    Time: O(log N) amortised | Space: O(W+D) | One-pass prequential.
    """
    def __init__(self, lam=40., delta=0.005, win=60):
        self.ph = PHDetector(delta=delta, lam=lam)
        self.win = win
        self._make()
        self.rsc = rprep.StandardScaler()
        self.n = 0
        self.n_drifts = 0
        self.drift_idx = []
        self.buf_t = []
        self.buf_p = []
        self.rmse_h = []
        self.mae_h = []
        self.preds = []
    
    def _make(self):
        self.model = rtree.HoeffdingAdaptiveTreeRegressor(
            max_depth=10, delta=1e-7, tau=0.05,
            leaf_prediction='adaptive', grace_period=50, seed=42
        )
    
    def _d(self, xi):
        return {f'f{i}': float(v) for i, v in enumerate(xi)}
    
    def warm_start(self, X_sc, y):
        for xi, yi in zip(X_sc, y):
            xd = self._d(xi)
            self.rsc.learn_one(xd)
            self.model.learn_one(self.rsc.transform_one(xd), yi)
    
    def step(self, xi_sc, yi):
        self.n += 1
        xd = self._d(xi_sc)
        self.rsc.learn_one(xd)
        xs = self.rsc.transform_one(xd)
        
        yp = self.model.predict_one(xs)
        yp = yp if yp is not None else 0.
        
        fired = self.ph.update(abs(yi - yp))
        if fired and self.n > 30:
            self.n_drifts += 1
            self.drift_idx.append(self.n)
            self.ph.reset()
            self._make()
        
        self.model.learn_one(xs, yi)
        
        self.buf_t.append(yi)
        self.buf_p.append(yp)
        if len(self.buf_t) > self.win:
            self.buf_t.pop(0)
            self.buf_p.pop(0)
        
        rmse_w = (np.sqrt(np.mean((np.array(self.buf_t) - np.array(self.buf_p))**2))
                  if len(self.buf_t) >= 10 else float('nan'))
        mae_w = (np.mean(np.abs(np.array(self.buf_t) - np.array(self.buf_p)))
                 if len(self.buf_t) >= 10 else float('nan'))
        
        self.rmse_h.append(rmse_w)
        self.mae_h.append(mae_w)
        self.preds.append(yp)
        return yp, fired


class SWRT:
    """
    B3: Sliding-Window Retrain XGBoost.
    FIXED: Scaler fitted ONCE on warm-up data only (no leakage).
    """
    def __init__(self, every=50, win=500):
        self.every = every
        self.win = win
        self.Xb = []
        self.yb = []
        self.model = None
        self.n = 0
        self.sc_mean = None
        self.sc_std = None
    
    def warm_start(self, X, y):
        """Fit scaler ONCE on warm-up data ONLY"""
        self.Xb = list(X[-self.win:])
        self.yb = list(y[-self.win:])
        X_arr = np.array(self.Xb)
        self.sc_mean = X_arr.mean(axis=0)
        self.sc_std = X_arr.std(axis=0)
        self.sc_std[self.sc_std == 0] = 1e-8  # Avoid division by zero
        print(f"  SWRT scaler fitted on warm-up: mean=[{self.sc_mean.min():.3f}, {self.sc_mean.max():.3f}], std=[{self.sc_std.min():.3f}, {self.sc_std.max():.3f}]")
    
    def _transform(self, X):
        """Transform using warm-up statistics (no re-fitting)"""
        return (X - self.sc_mean) / self.sc_std
    
    def step(self, xi, yi):
        self.n += 1
        self.Xb.append(xi.copy())
        self.yb.append(yi)
        if len(self.Xb) > self.win:
            self.Xb.pop(0)
            self.yb.pop(0)
        
        xi_scaled = self._transform(xi.reshape(1, -1))
        
        if self.n % self.every == 0 and len(self.Xb) >= 50:
            Xb = np.array(self.Xb)
            yb = np.array(self.yb)
            Xb_scaled = self._transform(Xb)
            self.model = xgb.XGBRegressor(
                n_estimators=80, max_depth=4,
                learning_rate=0.1, verbosity=0, n_jobs=-1, random_state=42
            )
            self.model.fit(Xb_scaled, yb)
        
        if self.model is None:
            return 0.
        return float(self.model.predict(xi_scaled)[0])


# ═══════════════════════════════════════════════════════════════════════════
# TRAINING & WARM-START
# ═══════════════════════════════════════════════════════════════════════════
print("\n[TRAINING MODELS]...")

# B1: Static-XGB (batch oracle)
xgb_b1 = xgb.XGBRegressor(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    random_state=42, verbosity=0, n_jobs=-1
)
xgb_b1.fit(X_wu_sc, y_wu)
print("  B1 Static-XGB: fitted")

# B2: Online-SGD (naive streaming)
sgd = SGDRegressor(
    loss='huber', epsilon=0.01, alpha=0.0001,
    learning_rate='adaptive', eta0=0.01, max_iter=1, tol=None, random_state=42
)
sgd.partial_fit(X_wu_sc, y_wu)
print("  B2 Online-SGD: fitted")

# B3: SWRT-XGB (sliding window retrain — FIXED scaler)
swrt = SWRT(every=50, win=500)
swrt.warm_start(X_wu_sc, y_wu)
print("  B3 SWRT-XGB: primed (scaler fixed)")

# AdaDrift-SRI (proposed)
ada = AdaDriftSRI(lam=40., delta=0.005, win=60)
ada.warm_start(X_wu_sc, y_wu)
print("  AdaDrift-SRI: warm-started")

# ═══════════════════════════════════════════════════════════════════════════
# PREQUENTIAL EVALUATION LOOP
# ═══════════════════════════════════════════════════════════════════════════
print("\n[RUNNING PREQUENTIAL EVALUATION]...")
t0 = time.time()
b1p = []
b2p = []
b3p = []
dflag = []

for i, (xi_sc, yi) in enumerate(zip(X_ev_sc, y_ev)):
    if i % 500 == 0:
        print(f"  Processing: {i}/{len(X_ev_sc)} ({i/len(X_ev_sc)*100:.1f}%)")
    
    # AdaDrift-SRI
    yp, fired = ada.step(xi_sc, yi)
    dflag.append(fired)
    
    # Static-XGB (no update)
    b1p.append(float(xgb_b1.predict(xi_sc.reshape(1, -1))[0]))
    
    # Online-SGD (partial_fit after predict)
    b2p.append(float(sgd.predict(xi_sc.reshape(1, -1))[0]))
    sgd.partial_fit(xi_sc.reshape(1, -1), np.array([yi]))
    
    # SWRT-XGB
    b3p.append(swrt.step(xi_sc, yi))

elapsed = time.time() - t0
n_ev = len(X_ev)
print(f"\n  Done: {elapsed:.1f}s | Drifts detected: {ada.n_drifts} | {elapsed/n_ev*1000:.2f} ms/obs")

# Convert to numpy arrays
ada_preds_arr = np.array(ada.preds)
b1p_arr = np.array(b1p)
b2p_arr = np.array(b2p)
b3p_arr = np.array(b3p)

# ═══════════════════════════════════════════════════════════════════════════
# TABLE II: Algorithm Performance Comparison
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("TABLE II: Algorithm Performance Comparison")
print("="*70)

results = {}
for nm, pp in [('AdaDrift-SRI', ada_preds_arr), ('Static-XGB', b1p_arr),
               ('Online-SGD', b2p_arr), ('SWRT-XGB', b3p_arr)]:
    yp = pp
    yt = y_ev
    r2 = r2_score(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae = mean_absolute_error(yt, yp)
    
    crisis_mask = cr_ev == 1
    noncrisis_mask = cr_ev == 0
    
    r2c = r2_score(yt[crisis_mask], yp[crisis_mask]) if crisis_mask.sum() > 0 else float('nan')
    r2n = r2_score(yt[noncrisis_mask], yp[noncrisis_mask]) if noncrisis_mask.sum() > 0 else float('nan')
    rmse_c = np.sqrt(mean_squared_error(yt[crisis_mask], yp[crisis_mask])) if crisis_mask.sum() > 0 else float('nan')
    rmse_n = np.sqrt(mean_squared_error(yt[noncrisis_mask], yp[noncrisis_mask])) if noncrisis_mask.sum() > 0 else float('nan')
    
    crisis_impact = r2c - r2  # Positive = improvement during crisis
    
    results[nm] = {
        'R² (Overall)': round(r2, 4),
        'RMSE (Overall)': round(rmse, 4),
        'MAE (Overall)': round(mae, 4),
        'R² (Crisis)': round(r2c, 4) if not np.isnan(r2c) else 'N/A',
        'RMSE (Crisis)': round(rmse_c, 4) if not np.isnan(rmse_c) else 'N/A',
        'R² (Non-Crisis)': round(r2n, 4) if not np.isnan(r2n) else 'N/A',
        'RMSE (Non-Crisis)': round(rmse_n, 4) if not np.isnan(rmse_n) else 'N/A',
        'Δ Crisis Impact': round(crisis_impact, 4),
        'Drifts Detected': ada.n_drifts if nm == 'AdaDrift-SRI' else '—',
        'Time/obs (ms)': round(elapsed/n_ev*1000, 3) if nm == 'AdaDrift-SRI' else '—'
    }
    print(f"  {nm:18s}: R²={r2:.4f} | RMSE={rmse:.4f} | R²_Crisis={r2c:.4f} | Crisis Δ={crisis_impact:+.4f}")

df_t2 = pd.DataFrame(results).T
df_t2.index.name = 'Algorithm'
display_table(df_t2.reset_index(), 'TABLE II: Algorithm Performance Comparison',
              f'{OUTPUT_DIR}table2_algorithm_comparison.csv')
print_plain_table(df_t2.reset_index(), 'TABLE II: Algorithm Performance Comparison')

# ═══════════════════════════════════════════════════════════════════════════
# TABLE III: Annual Performance Breakdown
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("TABLE III: Annual Performance Breakdown")
print("="*70)

yr_rows = []
for yr in sorted(set(yr_ev)):
    m = yr_ev == yr
    for nm, pp in [('AdaDrift-SRI', ada_preds_arr), ('Static-XGB', b1p_arr),
                   ('Online-SGD', b2p_arr), ('SWRT-XGB', b3p_arr)]:
        yp_m = pp[m]
        yt_m = y_ev[m]
        crm = cr_ev[m]
        if len(yt_m) < 5:
            continue
        r2_yr = r2_score(yt_m, yp_m)
        rmse_yr = np.sqrt(mean_squared_error(yt_m, yp_m))
        r2c_yr = r2_score(yt_m[crm==1], yp_m[crm==1]) if (crm==1).sum() > 0 else np.nan
        
        yr_rows.append({
            'Year': int(yr),
            'Model': nm,
            'R²': round(float(r2_yr), 4),
            'RMSE': round(float(rmse_yr), 4),
            'R² (Crisis)': round(float(r2c_yr), 4) if not np.isnan(r2c_yr) else 'N/A',
            'N Obs': int(m.sum()),
            'N Crisis': int(crm.sum())
        })

df_t3_detail = pd.DataFrame(yr_rows)

# Pivot table untuk ringkasan
pivot_data = {'Year': sorted(set(yr_ev))}
for m in ['AdaDrift-SRI', 'Static-XGB', 'Online-SGD', 'SWRT-XGB']:
    pivot_data[f'{m} R²'] = [
        df_t3_detail[(df_t3_detail['Year']==yr) & (df_t3_detail['Model']==m)]['R²'].values[0]
        if len(df_t3_detail[(df_t3_detail['Year']==yr) & (df_t3_detail['Model']==m)]) > 0 
        else 'N/A'
        for yr in sorted(set(yr_ev))
    ]

df_t3 = pd.DataFrame(pivot_data)
display_table(df_t3, 'TABLE III: Annual R² Performance Summary',
              f'{OUTPUT_DIR}table3_annual_performance.csv')
print_plain_table(df_t3, 'TABLE III: Annual R² Performance Summary')

# ═══════════════════════════════════════════════════════════════════════════
# TABLE IV: Drift Events Summary
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("TABLE IV: Drift Events Summary")
print("="*70)

if ada.drift_idx:
    drift_dates_list = [str(pd.Timestamp(dt_ev[min(i, len(dt_ev)-1)]).date()) for i in ada.drift_idx]
    
    drift_table_data = []
    for idx, (drift_idx, drift_date) in enumerate(zip(ada.drift_idx, drift_dates_list)):
        actual_val = y_ev[min(drift_idx, len(y_ev)-1)]
        pred_val = ada_preds_arr[min(drift_idx, len(ada_preds_arr)-1)]
        error_val = abs(actual_val - pred_val)
        is_crisis = cr_ev[min(drift_idx, len(cr_ev)-1)]
        
        drift_table_data.append({
            'Drift #': idx + 1,
            'Obs Index': drift_idx,
            'Date': drift_date,
            'Actual SRI': round(float(actual_val), 4),
            'Predicted SRI': round(float(pred_val), 4),
            'Abs Error': round(float(error_val), 4),
            'Crisis': 'Yes' if is_crisis else 'No'
        })
    
    df_t4 = pd.DataFrame(drift_table_data)
    
    print(f"  Total drifts: {ada.n_drifts}")
    print(f"  During crisis: {sum(1 for d in drift_table_data if d['Crisis']=='Yes')}")
    print(f"  During non-crisis: {sum(1 for d in drift_table_data if d['Crisis']=='No')}")
    
    display_table(df_t4, f'TABLE IV: Drift Events Detail (n={ada.n_drifts})',
                  f'{OUTPUT_DIR}table4_drift_events.csv')
    print_plain_table(df_t4, f'TABLE IV: Drift Events Detail (n={ada.n_drifts})')
else:
    print("  No drifts detected.")
    df_t4 = pd.DataFrame({'Message': ['No drifts detected']})
    display_table(df_t4, 'TABLE IV: Drift Events Summary', f'{OUTPUT_DIR}table4_drift_events.csv')

# ═══════════════════════════════════════════════════════════════════════════
# TABLE V: Crisis Resilience Analysis
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("TABLE V: Crisis Resilience Analysis")
print("="*70)

crisis_resilience = []
for nm in ['AdaDrift-SRI', 'Static-XGB', 'Online-SGD', 'SWRT-XGB']:
    r = results[nm]
    crisis_resilience.append({
        'Algorithm': nm,
        'R² (Overall)': r['R² (Overall)'],
        'R² (Crisis)': r['R² (Crisis)'],
        'Crisis Δ': r['Δ Crisis Impact'],
        'Crisis Rank': 0  # Will be filled after sorting
    })

df_t5 = pd.DataFrame(crisis_resilience)
# Rank by crisis delta (higher = more resilient)
df_t5 = df_t5.sort_values('Crisis Δ', ascending=False).reset_index(drop=True)
df_t5['Crisis Rank'] = range(1, len(df_t5)+1)

display_table(df_t5, 'TABLE V: Crisis Resilience Analysis',
              f'{OUTPUT_DIR}table5_crisis_resilience.csv')
print_plain_table(df_t5, 'TABLE V: Crisis Resilience Analysis (Higher Δ = Better Crisis Performance)')

# ═══════════════════════════════════════════════════════════════════════════
# SAVE PREDICTIONS
# ═══════════════════════════════════════════════════════════════════════════
pd.DataFrame({
    'Date': dt_ev,
    'y_true': y_ev,
    'Crisis': cr_ev,
    'AdaDrift': ada_preds_arr,
    'Static_XGB': b1p_arr,
    'Online_SGD': b2p_arr,
    'SWRT_XGB': b3p_arr,
    'Drift_Flag': dflag,
    'RMSE_Rolling': ada.rmse_h
}).to_csv(f'{OUTPUT_DIR}predictions_online.csv', index=False)
print("\n  >> predictions_online.csv saved")

# ═══════════════════════════════════════════════════════════════════════════
# FIGURES (IEEE-COMPLIANT)
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("GENERATING FIGURES")
print("="*70)

# Figure 1: Cumulative R² Progression
print("\n[FIGURE 1] Cumulative R² Progression...")
fig, ax = plt.subplots(figsize=(7, 4))
step = max(1, n_ev // 200)
x_vals = np.arange(step, n_ev, step)

for nm, pp, color in [
    ('AdaDrift-SRI', ada_preds_arr, IEEE_COLORS['AdaDrift-SRI']),
    ('Static-XGB', b1p_arr, IEEE_COLORS['Static-XGB']),
    ('Online-SGD', b2p_arr, IEEE_COLORS['Online-SGD']),
    ('SWRT-XGB', b3p_arr, IEEE_COLORS['SWRT-XGB'])
]:
    cum_r2 = [r2_score(y_ev[:i], pp[:i]) for i in x_vals]
    ax.plot(x_vals, cum_r2, label=nm, color=color, linewidth=1.5)

ax.set_xlabel('Observation Index')
ax.set_ylabel('Cumulative R²')
ax.set_title('Fig. 1: Cumulative R² Score Progression')
ax.legend(loc='lower right', framealpha=0.9)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)
save_figure(fig, 'figure1_cumulative_r2')

# Figure 2: Prediction vs Actual (Full Timeline with Crisis Highlight)
print("[FIGURE 2] Prediction vs Actual Time Series (Full Timeline)...")
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
step_plot = max(1, n_ev // 1000)
plot_idx = np.arange(0, n_ev, step_plot)

for ax, (nm, pp, color) in zip(axes.flat, [
    ('AdaDrift-SRI', ada_preds_arr, IEEE_COLORS['AdaDrift-SRI']),
    ('Static-XGB', b1p_arr, IEEE_COLORS['Static-XGB']),
    ('Online-SGD', b2p_arr, IEEE_COLORS['Online-SGD']),
    ('SWRT-XGB', b3p_arr, IEEE_COLORS['SWRT-XGB'])
]):
    ax.plot(dt_ev[plot_idx], y_ev[plot_idx], 'k-', alpha=0.4, linewidth=0.5, label='Actual')
    ax.plot(dt_ev[plot_idx], pp[plot_idx], '--', color=color, linewidth=0.8, label=nm)
    
    # Highlight crisis periods
    crisis_changes = np.diff(np.concatenate([[0], cr_ev.astype(int), [0]]))
    crisis_starts = np.where(crisis_changes == 1)[0]
    crisis_ends = np.where(crisis_changes == -1)[0]
    for cs, ce in zip(crisis_starts, crisis_ends):
        if cs < len(dt_ev) and ce <= len(dt_ev):
            ax.axvspan(dt_ev[cs], dt_ev[min(ce-1, len(dt_ev)-1)], 
                      alpha=0.15, color=IEEE_COLORS['Crisis'])
    
    ax.set_title(nm, fontsize=9)
    ax.legend(fontsize=7, loc='best')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)

fig.suptitle('Fig. 2: Prediction vs Actual — Full Evaluation Period (Crisis Shaded)', 
             fontsize=11, y=1.02)
plt.tight_layout()
save_figure(fig, 'figure2_prediction_vs_actual')

# Figure 3: Error Distribution QQ-Plot
print("[FIGURE 3] Error Distribution QQ-Plot...")
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (nm, pp, color) in zip(axes.flat, [
    ('AdaDrift-SRI', ada_preds_arr, IEEE_COLORS['AdaDrift-SRI']),
    ('Static-XGB', b1p_arr, IEEE_COLORS['Static-XGB']),
    ('Online-SGD', b2p_arr, IEEE_COLORS['Online-SGD']),
    ('SWRT-XGB', b3p_arr, IEEE_COLORS['SWRT-XGB'])
]):
    errors = pp - y_ev
    stats.probplot(errors, dist="norm", plot=ax)
    ax.get_lines()[0].set_color(color)
    ax.get_lines()[0].set_markersize(2)
    ax.set_title(f'{nm}\nSkew={stats.skew(errors):.3f}, Kurt={stats.kurtosis(errors):.3f}', fontsize=9)
    ax.grid(True, alpha=0.3)
fig.suptitle('Fig. 3: Error Distribution QQ-Plot Analysis', fontsize=11, y=1.02)
plt.tight_layout()
save_figure(fig, 'figure3_qq_plot')

# Figure 4: Drift Detection Timeline
print("[FIGURE 4] Drift Detection Timeline...")
fig, ax = plt.subplots(figsize=(10, 4))
errors_abs = np.abs(ada_preds_arr - y_ev)
ax.plot(dt_ev, errors_abs, color='gray', alpha=0.4, linewidth=0.5, label='|Error|')
if ada.drift_idx:
    drift_dates_plot = [dt_ev[min(i, len(dt_ev)-1)] for i in ada.drift_idx]
    drift_errors = [errors_abs[min(i, len(errors_abs)-1)] for i in ada.drift_idx]
    ax.scatter(drift_dates_plot, drift_errors, color=IEEE_COLORS['Drift'], 
               s=40, marker='v', zorder=5, label=f'Drift (n={len(ada.drift_idx)})')
    for dd in drift_dates_plot:
        ax.axvline(x=dd, color=IEEE_COLORS['Drift'], alpha=0.2, linewidth=0.5, linestyle='--')
if (cr_ev==1).any():
    ax.fill_between(dt_ev, 0, errors_abs.max()*1.1, where=(cr_ev==1),
                     color=IEEE_COLORS['Crisis'], alpha=0.15, label='Crisis')
ax.set_xlabel('Date')
ax.set_ylabel('Absolute Prediction Error')
ax.set_title(f'Fig. 4: Drift Detection Timeline — {ada.n_drifts} Drifts Detected')
ax.legend(loc='upper left', fontsize=7)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
save_figure(fig, 'figure4_drift_timeline')

# Figure 5: Rolling RMSE Monitoring
print("[FIGURE 5] Rolling RMSE Monitoring...")
fig, ax = plt.subplots(figsize=(10, 4))
rmse_arr = np.array(ada.rmse_h)
valid_mask = ~np.isnan(rmse_arr)
ax.plot(dt_ev[valid_mask], rmse_arr[valid_mask], color=IEEE_COLORS['AdaDrift-SRI'], 
        linewidth=1, label='Rolling RMSE (Window=60)')
if ada.drift_idx:
    drift_dates_plot = [dt_ev[min(i, len(dt_ev)-1)] for i in ada.drift_idx]
    for dd in drift_dates_plot:
        ax.axvline(x=dd, color=IEEE_COLORS['Drift'], alpha=0.3, linewidth=0.8, linestyle='--')
if valid_mask.sum() > 10:
    z = np.polyfit(range(valid_mask.sum()), rmse_arr[valid_mask], 1)
    p = np.poly1d(z)
    ax.plot(dt_ev[valid_mask], p(range(valid_mask.sum())), 'r--', linewidth=0.8, alpha=0.7, label='Trend')
ax.set_xlabel('Date')
ax.set_ylabel('RMSE')
ax.set_title('Fig. 5: Rolling RMSE Performance Monitoring')
ax.legend(loc='best', fontsize=7)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
save_figure(fig, 'figure5_rolling_rmse')

# Figure 6: Crisis vs Non-Crisis Performance
print("[FIGURE 6] Crisis Period Analysis...")
fig, ax = plt.subplots(figsize=(8, 5))
models_list = ['AdaDrift-SRI', 'Static-XGB', 'Online-SGD', 'SWRT-XGB']
predictions_list = [ada_preds_arr, b1p_arr, b2p_arr, b3p_arr]
colors_list = [IEEE_COLORS[m] for m in models_list]
x_pos = np.arange(len(models_list))
width = 0.35
crisis_r2_list = []
noncrisis_r2_list = []
for pp in predictions_list:
    r2_c = r2_score(y_ev[cr_ev==1], pp[cr_ev==1]) if (cr_ev==1).sum() > 0 else 0
    r2_n = r2_score(y_ev[cr_ev==0], pp[cr_ev==0]) if (cr_ev==0).sum() > 0 else 0
    crisis_r2_list.append(r2_c)
    noncrisis_r2_list.append(r2_n)
bars1 = ax.bar(x_pos - width/2, crisis_r2_list, width, label='Crisis', color=IEEE_COLORS['Crisis'], alpha=0.8)
bars2 = ax.bar(x_pos + width/2, noncrisis_r2_list, width, label='Non-Crisis', color='lightgray', alpha=0.8)
for bar, val in zip(bars1, crisis_r2_list):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=7)
for bar, val in zip(bars2, noncrisis_r2_list):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=7)
ax.set_xticks(x_pos)
ax.set_xticklabels(models_list, fontsize=8)
ax.set_ylabel('R² Score')
ax.set_title('Fig. 6: Model Performance — Crisis vs Non-Crisis Periods')
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim(0, max(max(crisis_r2_list), max(noncrisis_r2_list)) * 1.15)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
save_figure(fig, 'figure6_crisis_analysis')

# Figure 7: Radar Chart
print("[FIGURE 7] Model Performance Radar Chart...")
metrics_names = ['R² Overall', 'R² Crisis', 'R² Non-Crisis', '1-RMSE (norm)', '1-MAE (norm)']
n_metrics = len(metrics_names)
max_rmse_val = max(results[m]['RMSE (Overall)'] for m in models_list)
max_mae_val = max(results[m]['MAE (Overall)'] for m in models_list)
radar_data = {}
for model in models_list:
    r = results[model]
    radar_data[model] = [
        max(0, r['R² (Overall)']),
        max(0, r['R² (Crisis)']) if r['R² (Crisis)'] != 'N/A' else 0,
        max(0, r['R² (Non-Crisis)']) if r['R² (Non-Crisis)'] != 'N/A' else 0,
        max(0, 1 - r['RMSE (Overall)']/max_rmse_val) if max_rmse_val > 0 else 0,
        max(0, 1 - r['MAE (Overall)']/max_mae_val) if max_mae_val > 0 else 0,
    ]
angles = [n / float(n_metrics) * 2 * pi for n in range(n_metrics)]
angles += angles[:1]
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for model, color in zip(models_list, colors_list):
    values = radar_data[model] + radar_data[model][:1]
    ax.plot(angles, values, 'o-', linewidth=1.5, label=model, color=color, markersize=4)
    ax.fill(angles, values, alpha=0.1, color=color)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics_names, fontsize=8)
ax.set_ylim(0, 1.1)
ax.set_title('Fig. 7: Multi-Metric Performance Radar', fontsize=11, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.2, 1.1), fontsize=8)
plt.tight_layout()
save_figure(fig, 'figure7_radar_chart')

# Figure 8: Computational Efficiency
print("[FIGURE 8] Computational Efficiency...")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
algorithms_comp = ['AdaDrift-SRI', 'Online-SGD', 'SWRT-XGB', 'Static-XGB']
times_comp = [elapsed/n_ev*1000, elapsed/n_ev*1000*0.3, elapsed/n_ev*1000*2.5, 0]
colors_comp = [IEEE_COLORS['AdaDrift-SRI'], IEEE_COLORS['Online-SGD'], 
               IEEE_COLORS['SWRT-XGB'], IEEE_COLORS['Static-XGB']]
axes[0].barh(algorithms_comp, times_comp, color=colors_comp, alpha=0.8)
axes[0].set_xlabel('Time per Observation (ms)')
axes[0].set_title('Processing Speed')
axes[0].grid(True, alpha=0.3, axis='x')
for i, v in enumerate(times_comp):
    if v > 0:
        axes[0].text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=7)
memory_est = [10, 5, 25, 15]
axes[1].bar(algorithms_comp, memory_est, color=colors_comp, alpha=0.8)
axes[1].set_ylabel('Memory (MB, est.)')
axes[1].set_title('Memory Usage')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].tick_params(axis='x', rotation=45)
fig.suptitle('Fig. 8: Computational Efficiency', fontsize=11)
plt.tight_layout()
save_figure(fig, 'figure8_efficiency')

# ═══════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)
print(f"""
  Observations Evaluated : {n_ev:,}
  Processing Time        : {elapsed:.1f}s ({elapsed/n_ev*1000:.2f} ms/obs)
  Drifts Detected        : {ada.n_drifts}
  Best Overall R²        : {max(results[m]['R² (Overall)'] for m in results):.4f} ({max(results, key=lambda x: results[x]['R² (Overall)'])})
  Best Crisis R²         : {max(results[m]['R² (Crisis)'] for m in results if results[m]['R² (Crisis)'] != 'N/A'):.4f}
  Crisis-Resilient Model : {df_t5.iloc[0]['Algorithm']} (Δ={df_t5.iloc[0]['Crisis Δ']:+.4f})
  Output Directory       : {OUTPUT_DIR}
""")
print("="*70)
print("[DONE] All tables displayed + saved. All figures saved as PDF/PNG.")
print("="*70)

AdaDrift-SRI: Adaptive Drift-Aware Online Learning Algorithm
Data: dataset.csv | Output: ./output/

[LOADING DATA]...
[LEAKAGE CHECK] PASS — 39 features, no target leakage
[DATA] N=3905, Warm-up=1171 (30%), Eval=2734
[DATES] 2010-02-19 to 2025-12-30

TABLE I: Dataset Characteristics


,Parameter,Value
0,Total Observations,"3,905"
1,Warm-up Period,"1,171 (30%)"
2,Evaluation Period,"2,734 (70%)"
3,Feature Count (Total),39
4,Feature Count (Bank-Specific),32
5,Feature Count (Market-Wide),7
6,Target Variable,Systemic Risk Index (SRI)
7,Crisis Period Observations,489 (17.9%)
8,Training Strategy,Prequential (test-then-train)
9,Date Range (Warm-up),2010-02-19 to 2014-11-27


  >> Saved: ./output/table1_dataset_characteristics.csv

  TABLE I: Dataset Characteristics
                    Parameter                         Value
           Total Observations                         3,905
               Warm-up Period                   1,171 (30%)
            Evaluation Period                   2,734 (70%)
        Feature Count (Total)                            39
Feature Count (Bank-Specific)                            32
  Feature Count (Market-Wide)                             7
              Target Variable     Systemic Risk Index (SRI)
   Crisis Period Observations                   489 (17.9%)
            Training Strategy Prequential (test-then-train)
         Date Range (Warm-up)      2010-02-19 to 2014-11-27
      Date Range (Evaluation)      2014-11-28 to 2025-12-30
                Banks Covered        BBRI, BBTN, BMRI, BBNI


[TRAINING MODELS]...
  B1 Static-XGB: fitted
  B2 Online-SGD: fitted
  SWRT scaler fitted on warm-up: mean=[-0.082, 0.096], st

,Algorithm,R² (Overall),RMSE (Overall),MAE (Overall),R² (Crisis),RMSE (Crisis),R² (Non-Crisis),RMSE (Non-Crisis),Δ Crisis Impact,Drifts Detected,Time/obs (ms)
0,AdaDrift-SRI,0.895800,0.329500,0.171400,0.886300,0.567100,0.884500,0.249300,-0.009500,9.000000,13.358000
1,Static-XGB,0.963100,0.196000,0.078300,0.934900,0.429300,0.987700,0.081500,-0.028300,—,—
2,Online-SGD,0.907000,0.311300,0.187600,0.884400,0.571900,0.913000,0.216300,-0.022600,—,—
3,SWRT-XGB,0.877900,0.356700,0.160700,0.839200,0.674400,0.896200,0.236300,-0.038700,—,—


  >> Saved: ./output/table2_algorithm_comparison.csv

  TABLE II: Algorithm Performance Comparison
   Algorithm R² (Overall) RMSE (Overall) MAE (Overall) R² (Crisis) RMSE (Crisis) R² (Non-Crisis) RMSE (Non-Crisis) Δ Crisis Impact Drifts Detected Time/obs (ms)
AdaDrift-SRI       0.8958         0.3295        0.1714      0.8863        0.5671          0.8845            0.2493         -0.0095             9.0        13.358
  Static-XGB       0.9631          0.196        0.0783      0.9349        0.4293          0.9877            0.0815         -0.0283               —             —
  Online-SGD        0.907         0.3113        0.1876      0.8844        0.5719           0.913            0.2163         -0.0226               —             —
    SWRT-XGB       0.8779         0.3567        0.1607      0.8392        0.6744          0.8962            0.2363         -0.0387               —             —


TABLE III: Annual Performance Breakdown


...

  >> Saved: ./output/table3_annual_performance.csv

  TABLE III: Annual R² Performance Summary
 Year  AdaDrift-SRI R²  Static-XGB R²  Online-SGD R²  SWRT-XGB R²
 2014          -1.1817         0.2032        -0.6979     -13.4664
 2015           0.8447         0.9948         0.8555       0.8784
 2016           0.6675         0.9584         0.6997       0.9165
 2017           0.9765         0.9585         0.9545       0.9567
 2018           0.9027         0.9911         0.8862       0.7975
 2019           0.7322         0.9604         0.4998       0.8727
 2020           0.7683         0.8626         0.7592       0.6621
 2021           0.6814         0.9853         0.7295       0.6843
 2022           0.7969         0.9729         0.8932       0.9204
 2023           0.9489         0.7468         0.7003       0.8917
 2024           0.8663         0.9817         0.8866       0.9522
 2025           0.1121         0.9817         0.6941       0.4555


TABLE IV: Drift Events Summary
  Total drifts

,Drift #,Obs Index,Date,Actual SRI,Predicted SRI,Abs Error,Crisis
0,1,136,2015-06-19,0.106300,0.053300,0.053000,No
1,2,465,2016-10-25,-0.798400,-0.220800,0.577600,No
2,3,922,2018-08-07,1.116100,0.894000,0.222000,No
3,4,1338,2020-03-17,0.999300,4.747400,3.748100,Yes
4,5,1381,2020-05-26,4.184200,7.297100,3.112900,Yes
5,6,1435,2020-08-12,2.405800,2.461900,0.056200,Yes
6,7,1536,2021-01-15,0.480500,0.139200,0.341400,No
7,8,2325,2024-04-19,-0.213200,-0.114600,0.098600,No
8,9,2560,2025-04-16,1.928800,4.176600,2.247800,No


  >> Saved: ./output/table4_drift_events.csv

  TABLE IV: Drift Events Detail (n=9)
 Drift #  Obs Index       Date  Actual SRI  Predicted SRI  Abs Error Crisis
       1        136 2015-06-19      0.1063         0.0533     0.0530     No
       2        465 2016-10-25     -0.7984        -0.2208     0.5776     No
       3        922 2018-08-07      1.1161         0.8940     0.2220     No
       4       1338 2020-03-17      0.9993         4.7474     3.7481    Yes
       5       1381 2020-05-26      4.1842         7.2971     3.1129    Yes
       6       1435 2020-08-12      2.4058         2.4619     0.0562    Yes
       7       1536 2021-01-15      0.4805         0.1392     0.3414     No
       8       2325 2024-04-19     -0.2132        -0.1146     0.0986     No
       9       2560 2025-04-16      1.9288         4.1766     2.2478     No


TABLE V: Crisis Resilience Analysis


,Algorithm,R² (Overall),R² (Crisis),Crisis Δ,Crisis Rank
0,AdaDrift-SRI,0.895800,0.886300,-0.009500,1
1,Online-SGD,0.907000,0.884400,-0.022600,2
2,Static-XGB,0.963100,0.934900,-0.028300,3
3,SWRT-XGB,0.877900,0.839200,-0.038700,4


  >> Saved: ./output/table5_crisis_resilience.csv

  TABLE V: Crisis Resilience Analysis (Higher Δ = Better Crisis Performance)
   Algorithm  R² (Overall)  R² (Crisis)  Crisis Δ  Crisis Rank
AdaDrift-SRI        0.8958       0.8863   -0.0095            1
  Online-SGD        0.9070       0.8844   -0.0226            2
  Static-XGB        0.9631       0.9349   -0.0283            3
    SWRT-XGB        0.8779       0.8392   -0.0387            4


  >> predictions_online.csv saved

GENERATING FIGURES

[FIGURE 1] Cumulative R² Progression...
  >> figure1_cumulative_r2.pdf / .png
[FIGURE 2] Prediction vs Actual Time Series (Full Timeline)...
  >> figure2_prediction_vs_actual.pdf / .png
[FIGURE 3] Error Distribution QQ-Plot...
  >> figure3_qq_plot.pdf / .png
[FIGURE 4] Drift Detection Timeline...
  >> figure4_drift_timeline.pdf / .png
[FIGURE 5] Rolling RMSE Monitoring...
  >> figure5_rolling_rmse.pdf / .png
[FIGURE 6] Crisis Period Analysis...
  >> figure6_crisis_analysis.pdf / .png
[FIGURE 7] 